# 07 · 기아 종식은 언제? — 세계 기아 지수(GHI) 분석과 예측
교과서 **Ⅳ-02 인공지능 프로젝트 199~208쪽 예제**를 우리 수업 방식으로 옮겼습니다.
수행평가 ②의 «불러오기 → 전처리 → 모델 → 학습 → 평가» 네 단계와 **같은 순서**입니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

| 파일 | 내용 |
|---|---|
| `ghi.csv` | 나라별 세계 기아 지수 2000 · 2008 · 2015 · 2023 |
| `undernourished.csv` | 인구 중 영양실조 비율(%) |
| `stunting.csv` | 5세 미만 발육 저하 비율(%) |

## ① 데이터 불러오기

In [ ]:
주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ghi   = pd.read_csv(주소 + 'ghi.csv')
nouri = pd.read_csv(주소 + 'undernourished.csv')
stunt = pd.read_csv(주소 + 'stunting.csv')
ghi.head()

## ② 전처리
💡 이번 분석에 쓰지 않는 «2015년 이후 변화» 두 열을 지웁니다. 열을 지우는 것은 `axis=1`.

In [ ]:
ghi = ghi.drop(['Absolute change since 2015', '% change since 2015'], axis=____)
print('결측치\n', ghi.isnull().sum())
print('중복 행 :', ghi.duplicated().sum())
ghi.describe()

### 연도끼리의 상관 — 히트맵
나라 이름(`Country`)은 숫자가 아니라 상관계수를 못 구하니 빼고 그립니다.

In [ ]:
num = ghi.drop('Country', axis=1)
sns.heatmap(num.corr(), annot=True, cmap='Blues')
plt.title('GHI correlation by year')
plt.show()

### 세 데이터 합치기(통합)
열 이름이 셋 다 `2000, 2008 …` 로 같아서 그대로 붙이면 헷갈립니다. 이름을 바꾼 뒤 **옆으로**(`axis=1`) 붙입니다.
💡 빈칸은 평균으로 채웁니다 — `fillna(평균)`.

In [ ]:
yrs = ['2000', '2008', '2015', '2023']
g = num.rename(columns={y: 'GHI' + y for y in yrs})
n = nouri.drop('Country', axis=1).rename(columns={y: 'Nouri' + y for y in yrs})
s = stunt.drop('Country', axis=1).rename(columns={y: 'Stunt' + y for y in yrs})
df = pd.concat([g, n, s], axis=1)
df = df.fillna(df.____())
df.corr()[['GHI2023']].sort_values('GHI2023', ascending=False)

**생각해 보기 ①** 2023년 기아 지수와 가장 관련이 깊은 것은 영양실조 비율과 발육 저하 비율 중 무엇인가요? (______)

## ③ 모델 — 연도별 «평균 기아 지수»로 몇 년에 목표에 닿을지 예측
교과서는 네 해(2000·2008·2015·2023)의 **나라 평균 GHI** 를 입력(X)으로, **연도**를 정답(y)으로 두고 선형 회귀를 합니다.
«GHI가 10(보통 수준)이 되는 해는?», «0이 되는 해는?»을 묻기 위해서입니다.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = g.mean().values.reshape(-1, 1)        # 연도별 나라 평균 GHI (4행 1열)
y = np.array([2000, 2008, 2015, 2023])     # 정답: 연도
print(X.ravel().round(2))

model = LinearRegression()
model.____(X, y)
print('기울기 :', model.coef_, ' 절편 :', model.intercept_)

## ④ 예측과 평가
💡 예측은 `predict`. 입력은 «2차원»이라 `[[10]]` 처럼 대괄호 두 겹.

In [ ]:
print('GHI 10 이 되는 해 :', model.____([[10]]))
print('GHI  0 이 되는 해 :', model.predict([[0]]))
print('R² :', model.score(X, y))

plt.scatter(X, y, label='data')
plt.plot(X, model.predict(X), 'r-', label='linear regression')
plt.xlabel('mean GHI'); plt.ylabel('year'); plt.legend(); plt.show()

## 비판적으로 보기 (교과서 208~209쪽 «평가 결과를 반영해 개선»)
1. 점이 **네 개뿐**입니다. R²가 0.93으로 높게 나와도 믿을 만할까요?
2. GHI 0은 지금까지 한 번도 관측된 적 없는 값입니다. 데이터 **범위 밖**을 직선으로 늘려 맞히는 것(외삽)은 어떤 위험이 있을까요?
3. 나라마다 사정이 다른데 «평균» 하나로 묶었습니다. 어떤 나라가 가려질까요? — 우리 프로젝트의 **편향 점검**과 같은 질문입니다.

> 적어 보기:

---
### 정답 예시 (막혔을 때만)
- ② `axis=1` · 통합 `df.fillna(df.mean())` · ③ `model.fit(X, y)` · ④ `model.predict([[10]])`
- 기울기가 약 **−2.17**, 절편이 약 **2054** 로 나옵니다(교과서와 같음). GHI 10 → 약 2032년, 0 → 약 2054년.
- 생각해 보기 ①: 발육 저하·영양실조 모두 높지만 값은 실행 결과로 확인하세요.